# Pipelines and Components in Azure ML

# Notebook Setup

Set project paths and load workspace MLClient.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


In [2]:
import mlflow

In [3]:
pipelines_config_path = "assets/tutorials-materials/pipelines-configs"
pipelines_src_path = "assets/tutorials-materials/pipelines-configs/src"

# Building Azure ML Pipeline

In [4]:
experiment_name = "dmdp100-sample-pipelines-experiment"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='', creation_time=1763711996679, experiment_id='ab1358f1-3177-4422-85a3-6b31706b524a', last_update_time=None, lifecycle_stage='active', name='dmdp100-sample-pipelines-experiment', tags={}>

## Create the scripts

You'll build a pipeline with two steps:

1. **Prepare the data**: Fix missing data and normalize the data.
1. **Train the model**: Train a logistic regression model.

Run the following cells to create the **src** folder and the two scripts.

In [ ]:
%%writefile $pipelines_src_path/prep-data.py
# import libraries
import argparse
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

def main(args):
    # read data
    df = get_data(args.input_data)

    cleaned_data = clean_data(df)

    normalized_data = normalize_data(cleaned_data)

    output_df = normalized_data.to_csv((Path(args.output_data) / "diabetes.csv"), index = False)

# function that reads the data
def get_data(path):
    df = pd.read_csv(path)

    # Count the rows and print the result
    row_count = (len(df))
    print('Preparing {} rows of data'.format(row_count))
    
    return df

# function that removes missing values
def clean_data(df):
    df = df.dropna()
    
    return df

# function that normalizes the data
def normalize_data(df):
    scaler = MinMaxScaler()
    num_cols = ['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree']
    df[num_cols] = scaler.fit_transform(df[num_cols])

    return df

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--input_data", dest='input_data',
                        type=str)
    parser.add_argument("--output_data", dest='output_data',
                        type=str)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")

Writing artifacts/tutorials-materials/pipelines-configs/src/prep-data.py


In [ ]:
%%writefile $pipelines_src_path/train-model.py
# import libraries
import mlflow
import glob
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # enable autologging
    mlflow.autolog()

    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(data_path):

    all_files = glob.glob(data_path + "/*.csv")
    df = pd.concat((pd.read_csv(f) for f in all_files), sort=False)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    mlflow.sklearn.save_model(model, args.model_output)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png") 

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)
    parser.add_argument("--model_output", dest='model_output',
                        type=str)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Writing artifacts/tutorials-materials/pipelines-configs/src/train-model.py


## Define the components
To define the component you need to specify:

- **Metadata**: *name*, *display name*, *version*, *description*, *type* etc. The metadata helps to describe and manage the component.
- **Interface**: *inputs* and *outputs*. For example, a model training component will take training data and the regularization rate as input, and generate a trained model file as output. 
- **Command, code & environment**: the *command*, *code* and *environment* to run the component. Command is the shell command to execute the component. Code usually refers to a source code directory. Environment could be an AzureML environment (curated or custom created), docker image or conda environment.

Run the following cells to create a YAML for each component you want to run as a pipeline step.

In [39]:
%%writefile $pipelines_config_path/prep-data.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: prep_data
display_name: Prepare Training Data
version: 1
type: command
inputs:
  input_data: 
    type: uri_file
outputs:
  output_data:
    type: uri_folder
code: ./src
environment: azureml:dmdp100env@latest
command: >-
  python prep-data.py 
  --input_data ${{inputs.input_data}}
  --output_data ${{outputs.output_data}}

Overwriting artifacts/tutorials-materials/pipelines-configs/prep-data.yml


In [40]:
%%writefile $pipelines_config_path/train-model.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: train_model
display_name: Train a Logistic Regression Model
version: 1
type: command
inputs:
  training_data: 
    type: uri_folder
  reg_rate:
    type: number
    default: 0.01
outputs:
  model_output:
    type: mlflow_model
code: ./src
environment: azureml:dmdp100env@latest
command: >-
  python train-model.py 
  --training_data ${{inputs.training_data}} 
  --reg_rate ${{inputs.reg_rate}} 
  --model_output ${{outputs.model_output}} 

Overwriting artifacts/tutorials-materials/pipelines-configs/train-model.yml


Notice that the YAML file refers to the `.py` scripts we created. They need to be stored in the same directory as the YAML files under the `src` folder as specified in the `code` section of the YAML.

In [5]:
from azure.ai.ml import load_component

prep_data_component = load_component(source=pipelines_config_path + "/prep-data.yml")
train_model_component = load_component(source=pipelines_config_path + "/train-model.yml")

### Define component using `@command_component` decorator

In [47]:
!pip install mldesigner

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.6/168.6 kB 10.5 MB/s eta 0:00:00


In [5]:
%%writefile $pipelines_config_path/create-train-model-component-decorator.py

from mldesigner import command_component, Input, Output
import mlflow
import pandas as pd
import numpy as np
import glob
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

@command_component(
    name="train_model_decorator",
    display_name="Train a Logistic Regression Model with Decorator",
    version="1",
)
def train_model_func(
    training_data: Input(type="uri_folder"),
    model_output: Output(type="mlflow_model"),
    reg_rate: float = 0.01
):
    mlflow.autolog()

    # Load data
    all_files = glob.glob(training_data + "/*.csv")
    df = pd.concat((pd.read_csv(f) for f in all_files), sort=False)

    X = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure',
            'TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values
    y = df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=0
    )

    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    # Save to output directory
    mlflow.sklearn.save_model(model, model_output)

    # Evaluate
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test, y_scores[:,1])
    print("AUC:", auc)

    fpr, tpr, _ = roc_curve(y_test, y_scores[:,1])
    plt.plot([0, 1], [0, 1], "k--")
    plt.plot(fpr, tpr)
    plt.savefig("ROC-Curve.png")


Overwriting artifacts/tutorials-materials/pipelines-configs/create-train-model-component-decorator.py


In [ ]:
import importlib  
decor_components = importlib.import_module("assets.tutorials-materials.pipelines-configs.create-train-model-component-decorator")
ml_client.components.create_or_update(decor_components.train_model_func, version='2')

CommandComponent({'latest_version': None, 'intellectual_property': None, 'auto_increment_version': False, 'source': 'REMOTE.WORKSPACE.COMPONENT', 'is_anonymous': False, 'auto_delete_setting': None, 'name': 'train_model_decorator', 'description': None, 'tags': {'codegenBy': 'mldesigner'}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/components/train_model_decorator/versions/2', 'Resource__source_path': None, 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml._restclient.v2024_01_01_preview.models._models_py3.SystemData object at 0x788a1529b2b0>, 'serialize': <msrest.serialization.Serializer object at 0x788a1529af50>, 'command': "mldesigner execute --source create-train-model-component-decorator.py --name train_model_decorator 

In [ ]:
# !python $pipelines_config_path/create-train-model-component-decorator.py

## Register Components

To make the components accessible to other users in the workspace, you can also register components to the Azure Machine Learning workspace.

You can register a component with the code below. Note that some parameters are immutable after the first registration, such as envoronment. To update those parameters, you need to create a new version of the component by specifying the `version` parameter.

In [42]:
ml_client.components.create_or_update(prep_data_component, version='2')
ml_client.components.create_or_update(train_model_component, version='2')

CommandComponent({'latest_version': None, 'intellectual_property': None, 'auto_increment_version': False, 'source': 'REMOTE.WORKSPACE.COMPONENT', 'is_anonymous': False, 'auto_delete_setting': None, 'name': 'train_model', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/components/train_model/versions/2', 'Resource__source_path': None, 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml._restclient.v2024_01_01_preview.models._models_py3.SystemData object at 0x77995c8de290>, 'serialize': <msrest.serialization.Serializer object at 0x7799399557b0>, 'command': 'python train-model.py  --training_data ${{inputs.training_data}}  --reg_rate ${{inputs.reg_rate}}  --model_output ${{outputs.model_output}} ', '

## Load Registered Components

To use preciously registered components in your pipeline, you can load them from the workspace as shown below.

In [54]:
# ml_client.components.get(name="prep_data", version="2")
# prep_data_component = ml_client.components.get(name="prep_data", version="2")
train_model_decorator_component = ml_client.components.get(name="train_model_decorator", version="2")

## Download Registered Components

You can also download previously registered components from the workspace to use them in local development.

In [ ]:
# ml_client.components.download(
#     name="prep_data",
#     version="2",
#     download_path="assets/pipelines-configs/downloaded_components/prep_data"
# )

Method download: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Build the pipeline

After creating and loading the components, you can build the pipeline. You'll compose the two components into a pipeline. First, you'll want the `prep_data_component` component to run. The output of the first component should be the input of the second component `train_model_component`, which will train the model.

The `diabetes_classification` function represents the complete pipeline. The function expects one input variable: `pipeline_job_input`. A data asset was created during setup. You'll use the registered data asset as the pipeline input. 

In [55]:
from azure.ai.ml.dsl import pipeline

@pipeline()
def diabetes_classification_pipeline(pipeline_job_input):
    clean_data = prep_data_component(input_data=pipeline_job_input)
    # train_model = train_model_component(training_data=clean_data.outputs.output_data)
    train_model = train_model_decorator_component(training_data=clean_data.outputs.output_data)

    return {
        "pipeline_job_transformed_data": clean_data.outputs.output_data,
        "pipeline_job_trained_model": train_model.outputs.model_output,
    }


In [56]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

diabetes_data_input = Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data-file:1")

pipeline_job = diabetes_classification_pipeline(diabetes_data_input)

You can retrieve the configuration of the pipeline job by printing the `pipeline_job` object:

In [57]:
print(pipeline_job)

display_name: diabetes_classification_pipeline
type: pipeline
inputs:
  pipeline_job_input:
    type: uri_file
    path: azureml:diabetes-data-file:1
outputs:
  pipeline_job_transformed_data:
    type: uri_folder
  pipeline_job_trained_model:
    type: mlflow_model
jobs:
  clean_data:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_job_input}}
    outputs:
      output_data: ${{parent.outputs.pipeline_job_transformed_data}}
    component:
      $schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
      name: prep_data
      version: '1'
      display_name: Prepare Training Data
      type: command
      inputs:
        input_data:
          type: uri_file
      outputs:
        output_data:
          type: uri_folder
      command: python prep-data.py  --input_data ${{inputs.input_data}} --output_data
        ${{outputs.output_data}}
      environment: azureml:/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resour

You can change any parameter of the pipeline job configuration by referring to the parameter and specifying the new value:

In [58]:
# change the output mode
pipeline_job.outputs.pipeline_job_transformed_data.mode = "upload"
pipeline_job.outputs.pipeline_job_trained_model.mode = "upload"
# set pipeline level compute
pipeline_job.settings.default_compute = "dmdp100-cpu-cluster"
# set pipeline level datastore
pipeline_job.settings.default_datastore = "dmdp100"

# set experiment name
pipeline_job.experiment_name = experiment_name
# print the pipeline job again to review the changes
print(pipeline_job)

display_name: diabetes_classification_pipeline
type: pipeline
inputs:
  pipeline_job_input:
    type: uri_file
    path: azureml:diabetes-data-file:1
outputs:
  pipeline_job_transformed_data:
    mode: upload
    type: uri_folder
  pipeline_job_trained_model:
    mode: upload
    type: mlflow_model
jobs:
  clean_data:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_job_input}}
    outputs:
      output_data: ${{parent.outputs.pipeline_job_transformed_data}}
    component:
      $schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
      name: prep_data
      version: '1'
      display_name: Prepare Training Data
      type: command
      inputs:
        input_data:
          type: uri_file
      outputs:
        output_data:
          type: uri_folder
      command: python prep-data.py  --input_data ${{inputs.input_data}} --output_data
        ${{outputs.output_data}}
      environment: azureml:/subscriptions/d34fa9f4-

## Submit the pipeline job

Finally, when you've built the pipeline and configured the pipeline job to run as required, you can submit the pipeline job:

In [59]:
# submit job to workspace
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name=experiment_name
)
pipeline_job

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored


Experiment,Name,Type,Status,Details Page
dmdp100-sample-pipelines-experiment,shy_energy_8w0wffv414,pipeline,NotStarted,Link to Azure Machine Learning studio


# Schedule a pipeline job 


To schedule a pipeline job, you need to use the `JobSchedule` class to associate a schedule to a pipeline job.

In [ ]:
from azure.ai.ml.entities import RecurrenceTrigger, RecurrencePattern
from azure.ai.ml.constants import TimeZone
from datetime import datetime as dt, timezone, timedelta

# Now
# schedule_start_time = dt.now(timezone.utc)
# tomorrow at 10:00 AM UTC
now = dt.now(timezone.utc)
schedule_start_time = (now + timedelta(days=1)).replace(hour=10, minute=0, second=0, microsecond=0)
# iso_start_time = schedule_start_time.isoformat().replace("+00:00", "Z")
iso_start_time = schedule_start_time.strftime("%Y-%m-%dT%H:%M:%SZ")

recurrence_trigger = RecurrenceTrigger(
    frequency="month",
    interval=1,
    # schedule=RecurrencePattern(month_days=[1], hours=[10], minutes=[0]), # will run on the 1st of every month at 10:00 AM UTC
    schedule=RecurrencePattern(month_days=[1], hours=[10], minutes=[0, 1]), # will run on the 1st of every month at 10:00 AM UTC and at 10:01 AM UTC
    # start_time=iso_start_time,
    time_zone=TimeZone.UTC,
)

In [ ]:
from azure.ai.ml.entities import JobSchedule


schedule_name = "diabetes_classification_pipeline_test_monthly_schedule"
job_schedule = JobSchedule(
    name=schedule_name, trigger=recurrence_trigger, create_job=pipeline_job
)

job_schedule = ml_client.schedules.begin_create_or_update(
    schedule=job_schedule
).result()

## Disable a schedule

In [52]:
job_schedule = ml_client.schedules.begin_disable(name=schedule_name).result()
job_schedule.is_enabled

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored


..

False

## Delete a schedule

In [53]:
# Only disabled schedules can be deleted
ml_client.schedules.begin_delete(name=schedule_name).result()

.................